<a href="https://colab.research.google.com/github/Bartimeyss/Ml-contest/blob/main/ml_kontest_yo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"嗨，毫升"

'嗨，毫升'

# Import modules and data

In [5]:
import numpy as np
import pandas as pd
import torch as trc

In [8]:

import os

#download files

os.environ['KAGGLE_USERNAME'] = "nezeritka"
os.environ['KAGGLE_KEY'] = "4da40b42a8f49222ac788d2832884481"
!kaggle competitions download -c scooter-price-prediction-edu-2026
!tar -xf scooter-price-prediction-edu-2026.zip
!del  scooter-price-prediction-edu-2026.zip



scooter-price-prediction-edu-2026.zip: Skipping, found more recently modified local copy (use --force to force download)


In [9]:
# Проверка загрузки

train_path = f"train.csv"
test_path = f"test.csv"
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)




In [10]:
#train
print(test.head())
print(train.head())

   id  trip_duration_min  distance_km  battery_level_start  temperature_c  \
0   0          11.328001     2.984336            55.779495       8.091798   
1   1          38.498660    11.646348            82.460577      14.587610   
2   2          12.336503     3.903070            27.184917      26.280163   
3   3          18.232429     5.046844            25.686523            NaN   
4   4          22.992648     8.149461            39.617766       9.642400   

   wind_speed  demand_index  distance_km_noisy city_zone scooter_model  \
0    4.066090      0.663601           1.963757    center             B   
1    0.324484      0.025097          10.144005    suburb             B   
2    0.803103      0.550207           0.263852      park             A   
3    9.745415      0.365754           7.439801    center             B   
4    4.688548      0.991382           6.484963    suburb             B   

   is_weekend  avg_price_last_week  route_complexity  driver_experience  \
0           0    

In [61]:
train.columns

Index(['id', 'trip_duration_min', 'distance_km', 'battery_level_start',
       'temperature_c', 'wind_speed', 'demand_index', 'distance_km_noisy',
       'city_zone', 'scooter_model', 'is_weekend', 'rental_price',
       'avg_price_last_week', 'route_complexity', 'driver_experience',
       'weather_rating'],
      dtype='str')

In [94]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

features = ['id', 'trip_duration_min', 'distance_km', 'battery_level_start',
       'temperature_c', 'wind_speed', 'demand_index', 'distance_km_noisy', 'is_weekend',
       'avg_price_last_week', 'route_complexity', 'driver_experience',
       'weather_rating']

target = 'rental_price'  

ids = train['id']
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(train[features], train[target], ids, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
# Инициализация и обучение модели
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.1,
    depth=8,
    verbose=100,
    loss_function='RMSE'
)

model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

print("Обучение завершено.")

0:	learn: 5.4427339	test: 5.4112280	best: 5.4112280 (0)	total: 7.36ms	remaining: 14.7s
100:	learn: 1.2752564	test: 2.4122329	best: 2.3950832 (90)	total: 668ms	remaining: 12.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.395083239
bestIteration = 90

Shrink model to first 91 iterations.
Обучение завершено.


In [95]:
y_pred = model.predict(X_test)

In [96]:
r2_score(y_pred, y_test)

0.7725979140861394

0.7619595731973333

-0.5649862240280243

# Process data



In [ ]:
from sklearn.preprocessing import OneHotEncoder

train = train.drop(['id'])
test = train.drop(['id'])
categorical_columns = ['city_zone', 'scooter_model']
encoder = OneHotEncoder(sparse_output=True)


# Gridsearch or optuna ig